# Caption pipeline validation (Colab T4)

End-to-end check of the Mefid captioning pipeline on GPU before Docker/RunPod deployment.

**Pipeline:** scene detect → token budget → frame extract → window → Qwen2-VL-2B → merge gate

**Runtime:** Colab → **Change runtime type → T4 GPU**

**Video:** upload a test clip below (recommended) or point at a file you copied to Drive.

In [7]:
import torch

assert torch.cuda.is_available(), "Switch runtime to GPU (T4) before continuing."
print(f"CUDA: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

CUDA: Tesla T4
VRAM: 15.6 GB


## 1. Get the repo

Pick **one** path:

- **A (recommended):** Upload the `mefid` folder to Google Drive, mount Drive, set `REPO_ROOT` below.
- **B:** `git clone` your repo into `/content/mefid` (uncomment and set URL).

In [8]:
from pathlib import Path

# --- Option A: Drive mount (uncomment if using Drive) ---
from google.colab import drive
drive.mount("/content/drive")
REPO_ROOT = Path("/content/drive/MyDrive/mefid")

# --- Option B: git clone (uncomment and set your URL) ---
# !git clone https://github.com/YOUR_USER/mefid.git /content/mefid
# REPO_ROOT = Path("/content/mefid")

# --- Option C: already in Colab after uploading repo zip ---
# REPO_ROOT = Path("/content/mefid")

assert (REPO_ROOT / "services/caption/src/caption_service.py").exists(), (
    f"Repo not found at {REPO_ROOT}. Mount Drive, clone, or upload the mefid folder."
)
print(f"Repo: {REPO_ROOT}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Repo: /content/drive/MyDrive/mefid


In [ ]:
import os
import subprocess
import sys

os.chdir(REPO_ROOT)

# Set before any `exports.schema.constants` import (T4 has ~15 GB VRAM).
# Model loads in 4-bit on CUDA; these tune per-window frame count / resolution.

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "services/exports"])
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "-e", "services/caption[gpu]"]
)

# Match local test layout: services/caption + exports on sys.path
sys.path.insert(0, str(REPO_ROOT / "services/caption"))
sys.path.insert(0, str(REPO_ROOT / "services/exports/src"))

print("Dependencies installed.")

Dependencies installed.
T4 tunables: CAPTION_TARGET_FPS=2.0  CAPTION_MAX_PIXELS_DEFAULT=401408


## 2. Test video

Upload a short highlight clip (e.g. Saibari goal). Colab cannot reach a local MinIO on your laptop — use a file upload or a publicly reachable object URL.

In [ ]:
from pathlib import Path
from uuid import UUID

from google.colab import files

MEDIA_ID = UUID("a0829b5c-0000-4000-8000-000000000001")  # placeholder for validation
VIDEO_PATH = Path("/content/drive/MyDrive/mefid/videos/new_signing.mp4")

if not VIDEO_PATH.exists():
    print("Upload a .mp4 test clip:")
    uploaded = files.upload()
    name = next(iter(uploaded))
    VIDEO_PATH = Path("/content") / name
    print(f"Saved to {VIDEO_PATH}")

assert VIDEO_PATH.exists(), f"Video not found: {VIDEO_PATH}"
print(f"media_id={MEDIA_ID}  video={VIDEO_PATH}  size={VIDEO_PATH.stat().st_size / 1e6:.1f} MB")

## 3. Dry run — scenes, fps, windows (no VLM)

Quick sanity check before loading Qwen (~2 GB download).

In [ ]:
from src.pipeline.scene_detection import detect_scenes
from src.pipeline.token_budget import compute_fps_and_resolution
from src.pipeline.frame_extraction import extract_scene_frames
from src.pipeline.windowing import build_windows

scenes = detect_scenes(str(VIDEO_PATH))
print(f"Scenes: {len(scenes)}\n")

total_windows = 0
for i, scene in enumerate(scenes):
    fps, max_pixels = compute_fps_and_resolution(scene.duration)
    frames, timestamps = extract_scene_frames(str(VIDEO_PATH), scene, fps)
    windows = build_windows(frames, timestamps)
    total_windows += len(windows)
    print(
        f"scene {i:02d}  {scene.start_time:7.2f}s–{scene.end_time:7.2f}s  "
        f"dur={scene.duration:5.1f}s  fps={fps:.2f}  max_pixels={max_pixels}  "
        f"frames={len(frames)}  windows={len(windows)}"
    )

print(f"\nTotal VLM calls (pre-merge): {total_windows}")

## 4. Load Qwen2-VL-2B and run full pipeline

First run downloads model weights from Hugging Face.



In [ ]:
import time

from src.caption_service import caption_video
from src.pipeline.caption_generation import CaptionGenerator

t0 = time.perf_counter()
generator = CaptionGenerator.load()
print(f"Model loaded in {time.perf_counter() - t0:.1f}s")

t1 = time.perf_counter()
captions = caption_video(str(VIDEO_PATH), MEDIA_ID, generator)
elapsed = time.perf_counter() - t1
print(f"\n{caption_video.__name__} finished in {elapsed:.1f}s — {len(captions)} captions")

In [ ]:
import pandas as pd

rows = [
    {
        "start": f"{c.start_time:.2f}",
        "end": f"{c.end_time:.2f}",
        "dur": f"{c.end_time - c.start_time:.1f}s",
        "text": c.text,
    }
    for c in captions
]
df = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", 120)
display(df)

print("\n--- Full text ---")
for i, c in enumerate(captions):
    print(f"\n[{i:02d}] {c.start_time:.2f}s – {c.end_time:.2f}s\n{c.text}")